In [ ]:
"""
=========================================================
09_create_deciles.py
=========================================================

Purpose
-------
Create propensity deciles and randomly assign customers
to Control and Treatment groups.

Input
-----
outputs/customer_propensity_scores.csv

Outputs
-------
outputs/campaign_customers.csv
outputs/decile_summary.csv
"""

In [ ]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")

In [ ]:
#Global declare random state
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# ==========================================================
# Paths
# ==========================================================

BASE_DIR = Path.cwd().parent

In [ ]:
print("=" * 70)
print("CREATING PROPENSITY DECILES")
print("=" * 70)

##########################################################
# LOAD SCORED CUSTOMERS
##########################################################

df = pd.read_csv(
    f"{BASE_DIR}/outputs/customer_propensity_scores.csv"
)

In [ ]:
##########################################################
# CREATE DECILES
##########################################################

# Highest propensity should be Decile 10

df["Propensity_Decile"] = pd.qcut(
    df["propensity_score"],
    10,
    labels=False,
    duplicates="drop"
) + 1

In [ ]:
##########################################################
# MAKE DECILE 10 = HIGHEST PROPENSITY
##########################################################

df["Propensity_Decile"] = 11 - df["Propensity_Decile"]

In [ ]:
##########################################################
# RANDOM CONTROL / TREATMENT SPLIT
##########################################################

def split_group(group):

    group = group.sample(
        frac=1,
        random_state=RANDOM_STATE
    ).reset_index(drop=True)

    midpoint = len(group) // 2

    group.loc[:midpoint-1, "Group"] = "Control"
    group.loc[midpoint:, "Group"] = "Treatment"

    return group


df = (

    df

    .groupby("Propensity_Decile", group_keys=False)

    .apply(split_group)

)

In [ ]:
##########################################################
# VALIDATION
##########################################################

print("\n")

print("="*60)
print("CUSTOMERS PER DECILE")
print("="*60)

print(

    df["Propensity_Decile"]

    .value_counts()

    .sort_index()

)

print("\n")

print("="*60)
print("CONTROL / TREATMENT SPLIT")
print("="*60)

split = pd.crosstab(

    df["Propensity_Decile"],

    df["Group"]

)

print(split)

In [ ]:
##########################################################
# AVERAGE PROPENSITY
##########################################################

print("\n")

print("="*60)
print("AVERAGE PROPENSITY SCORE")
print("="*60)

summary = (

    df

    .groupby(["Propensity_Decile","Group"])

    ["propensity_score"]

    .agg(

        Customers="count",

        Mean="mean",

        Min="min",

        Max="max"

    )

    .reset_index()

)

print(summary)

In [ ]:
##########################################################
# SAVE SUMMARY
##########################################################

summary.to_csv(

    f"{BASE_DIR}/outputs/decile_summary.csv",

    index=False

)

In [ ]:
##########################################################
# SAVE CAMPAIGN FILE
##########################################################

campaign = df[[

    "Client_Code",

    "propensity_score",

    "Propensity_Decile",

    "Group"

]].copy()

campaign.to_csv(

    f"{BASE_DIR}/outputs/campaign_customers.csv",

    index=False

)

In [ ]:
##########################################################
# FINAL MESSAGE
##########################################################

print("\n")

print("="*70)

print("Campaign file successfully created.")

print()

print(f"Total Customers : {len(campaign):,}")

print(f"Control Group   : {(campaign['Group']=='Control').sum():,}")

print(f"Treatment Group : {(campaign['Group']=='Treatment').sum():,}")

print()

print("Files Saved")

print(f"{BASE_DIR}/outputs/campaign_customers.csv")

print(f"{BASE_DIR}/outputs/decile_summary.csv")

print("="*70)